In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import torch
import random
import pickle
import numpy as np
from itertools import product

In [3]:
DATA_DIR="/content/drive/MyDrive/drug_repurposing/data"

In [7]:
!pip install torch_geometric

In [9]:
checkpoint = torch.load(f"{DATA_DIR}/processed/filtered_graph.pt", weights_only=False)

In [10]:
data=checkpoint["data"]
node_maps=checkpoint["node_maps"]
reverse_node_maps=checkpoint["reverse_node_maps"]

In [11]:
ctd_edge_index=data[("Compound", "CtD", "Disease")].edge_index

#list of drug index and disease index
positive_edges=list(zip(
    ctd_edge_index[0].tolist(),
    ctd_edge_index[1].tolist()
))

len(positive_edges)

487

In [13]:
#shuffle
random.seed(42)
random.shuffle(positive_edges)
num_edges = len(positive_edges)

In [14]:
#split
train_ratio=0.7
val_ratio=0.15

train_end=int(train_ratio * num_edges)
val_end=int((train_ratio + val_ratio) * num_edges)

train_pos=positive_edges[:train_end]
val_pos=positive_edges[train_end:val_end]
test_pos=positive_edges[val_end:]

len(train_pos), len(val_pos), len(test_pos)

(340, 73, 74)

In [15]:
train_pos_set = set(train_pos)
val_pos_set = set(val_pos)
test_pos_set = set(test_pos)

all_pos_set = set(positive_edges)

In [16]:
#define negative space
num_drugs = data["Compound"].num_nodes
num_diseases = data["Disease"].num_nodes

all_pairs = set(product(range(num_drugs), range(num_diseases)))

In [17]:
candidate_negatives = list(all_pairs - all_pos_set)
len(candidate_negatives)

192607

In [18]:
#negative sampling
def sample_negatives(pos_edges, ratio=1):
    num_neg = len(pos_edges) * ratio
    return random.sample(candidate_negatives, num_neg)

In [19]:
train_neg=sample_negatives(train_pos, ratio=1)
val_neg=sample_negatives(val_pos, ratio=1)
test_neg=sample_negatives(test_pos, ratio=1)

len(train_neg), len(val_neg), len(test_neg)

(340, 73, 74)

In [21]:
#leakage?
assert len(set(train_neg) & all_pos_set) == 0
assert len(set(val_neg) & all_pos_set) == 0
assert len(set(test_neg) & all_pos_set) == 0

print("no leakage detected")

no leakage detected


In [22]:
OUTPUT_DIR=f"{DATA_DIR}/processed/edge_splits"
NEG_DIR=f"{DATA_DIR}/processed/negative_samples"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(NEG_DIR, exist_ok=True)

with open(f"{OUTPUT_DIR}/train_pos.pkl", "wb") as f:
    pickle.dump(train_pos, f)

with open(f"{OUTPUT_DIR}/val_pos.pkl", "wb") as f:
    pickle.dump(val_pos, f)

with open(f"{OUTPUT_DIR}/test_pos.pkl", "wb") as f:
    pickle.dump(test_pos, f)

with open(f"{NEG_DIR}/train_neg.pkl", "wb") as f:
    pickle.dump(train_neg, f)

with open(f"{NEG_DIR}/val_neg.pkl", "wb") as f:
    pickle.dump(val_neg, f)

with open(f"{NEG_DIR}/test_neg.pkl", "wb") as f:
    pickle.dump(test_neg, f)